# Browser Automation with Selenium

`requests` + BeautifulSoup only see raw HTML. When a page builds its content with **JavaScript** (infinite scrolls, dashboards, logins, games...), the data is created *after* the page loads — so we need a real browser.

**Selenium** drives a real browser (Chrome, Firefox, ...) programmatically: it opens pages, clicks buttons, fills forms, and reads the rendered results — exactly like a human, but automated.

**What you will learn:**

- Launch Chrome with Selenium and navigate to pages
- Find elements with `By.ID`, `By.CLASS_NAME`, `By.CSS_SELECTOR`, `By.XPATH`
- Click, type, and submit forms
- Wait for dynamic content (`WebDriverWait` instead of `sleep`)
- Build a safe login workflow — **without ever writing real credentials in code**
- Automate a real game (Cookie Clicker) with a *bounded* loop

> **Setup:** `pip install selenium`. Modern Selenium (4.6+) includes **Selenium Manager**, which downloads the matching `chromedriver` automatically — no manual driver setup needed.

## Starting the Browser

`webdriver.Chrome()` opens a real Chrome window. Everything the browser does from here on is controlled by our code.

**Golden rule:** always `driver.quit()` when done — otherwise orphan Chrome processes pile up. `try/finally` guarantees this even when code crashes.

In [ ]:
from selenium import webdriver

# Headless mode (no window) — useful on servers:
# options = webdriver.ChromeOptions()
# options.add_argument("--headless")
# driver = webdriver.Chrome(options=options)

driver = webdriver.Chrome()
try:
    driver.get("https://www.scrapethissite.com/pages/simple/")
    print("Page title:", driver.title)
    print("Current URL:", driver.current_url)
finally:
    driver.quit()   # ALWAYS close the browser

## Finding Elements

Selenium finds elements by **strategy**:

| Strategy | Example | Best for |
|----------|---------|----------|
| `By.ID` | `find_element(By.ID, "q")` | Unique element (forms, buttons) |
| `By.CLASS_NAME` | `find_element(By.CLASS_NAME, "country-name")` | Repeated blocks |
| `By.CSS_SELECTOR` | `find_element(By.CSS_SELECTOR, "h3.country-name")` | CSS-style queries |
| `By.XPATH` | `find_element(By.XPATH, "//div[@id='bigCookie']")` | Complex navigation |
| `By.NAME` | `find_element(By.NAME, "q")` | Form fields |

- `find_element(...)` — first match (raises if none)
- `find_elements(...)` — list of all matches (empty if none)

In [ ]:
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
try:
    driver.get("https://www.scrapethissite.com/pages/simple/")

    # Same selectors as BeautifulSoup — but on the LIVE rendered page
    country_names = driver.find_elements(By.CLASS_NAME, "country-name")
    print("Countries found:", len(country_names))
    for el in country_names[:5]:
        print(" -", el.text.strip())
finally:
    driver.quit()

## Interacting with a Page

Three operations cover most automation:

- `element.click()` — click a button / link
- `element.send_keys("text")` — type into an input
- `element.submit()` — submit the enclosing form

The practice site's "forms" page has a search box (`id="q"`). Let's type a team name, submit, and read the results — **waiting** for the page to update with `WebDriverWait` + `expected_conditions` instead of blind `time.sleep()`:

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
try:
    driver.get("https://www.scrapethissite.com/pages/forms/")

    # Type a query into the search box and submit the form
    search_box = driver.find_element(By.ID, "q")
    search_box.send_keys("Boston")
    search_box.submit()

    # Wait up to 10 seconds for the results table to appear
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
    print("Result rows:", len(rows))
    for row in rows[:3]:
        print(" -", row.text.replace("\t", " | "))
finally:
    driver.quit()

## The Login Workflow (the safe way)

Logging in is just: **fill email → fill password → click submit**. But a course notebook must never contain real credentials — the notebook ends up in git, and passwords leak.

**Rules for credentials:**
1. Never hardcode emails or passwords in code
2. Ask for them at runtime with `input()` / `getpass` (hidden input)
3. Or read them from environment variables

The selector names below are placeholders — adapt them to the site you own (you may need to inspect the page with the browser's DevTools):

In [ ]:
from getpass import getpass

# Ask for credentials at runtime — NOTHING is stored in the notebook
email = input("Email: ")
password = getpass("Password: ")   # typed input is hidden

driver = webdriver.Chrome()
try:
    driver.get("https://example.com/login")  # replace with YOUR site

    driver.find_element(By.ID, "email").send_keys(email)
    driver.find_element(By.ID, "password").send_keys(password)
    driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()

    # Wait for something that only exists AFTER login, e.g. a dashboard element
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "user-profile"))
    )
    print("Login successful — dashboard reached.")
finally:
    driver.quit()

## Waits: `sleep` vs `WebDriverWait`

| Approach | Problem |
|----------|---------|
| `time.sleep(5)` | Wastes time when the page is fast; still fails when it's slow (fixed delay) |
| `WebDriverWait(driver, 10).until(EC....)` | Waits **exactly as long as needed**, up to a timeout |

Use `WebDriverWait` + `expected_conditions` whenever the page changes *after* loading (AJAX, filters, SPAs). Common conditions:

- `EC.presence_of_element_located(...)` — element exists in the DOM
- `EC.element_to_be_clickable(...)` — element is visible and clickable
- `EC.visibility_of_element_located(...)` — element is visible

## Case Study: Automating a Game

Cookie Clicker builds the whole game with JavaScript — a perfect Selenium target. We click the big cookie and buy upgrades.

> **Important:** the original version of this cell used `while True:` — an **infinite loop** that locks the notebook forever. Always bound your loops with a click limit!

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
try:
    driver.get("https://orteil.dashnet.org/cookieclicker/")

    # Dismiss the language-select overlay if it appears (JS-rendered UI)
    try:
        WebDriverWait(driver, 15).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div.langSelectButton"))
        ).click()
    except Exception:
        pass  # overlay never appeared — that's fine

    # Wait for the game to finish loading, then grab the cookie
    # (the game is heavy — allow up to 60 seconds)
    cookie = WebDriverWait(driver, 60).until(
        EC.presence_of_element_located((By.ID, "bigCookie"))
    )

    clicks = 0
    MAX_CLICKS = 100  # bounded loop — never use while True here!

    while clicks < MAX_CLICKS:
        # Re-find the cookie every click: the game re-renders the DOM
        # constantly, which makes saved element references go stale.
        try:
            driver.find_element(By.ID, "bigCookie").click()
        except Exception:
            pass
        clicks += 1

        # Every 20 clicks, buy whatever upgrade/product is enabled
        if clicks % 20 == 0:
            for upgrade in driver.find_elements(
                By.XPATH, "//div[contains(@class,'enabled')]"
            ):
                try:
                    upgrade.click()
                except Exception:
                    pass

    print(f"Clicked {clicks} times — the game ran for us!")
finally:
    driver.quit()

## Best Practices

- **Always** `driver.quit()` — wrap sessions in `try/finally`.
- Prefer `WebDriverWait` + `expected_conditions` over `time.sleep()`.
- Never hardcode credentials — use `input()`, `getpass`, or environment variables.
- Bound every loop with a counter or condition — never `while True` in a notebook.
- Prefer robust selectors (`By.ID`, stable classes) over fragile XPath indexes.
- Re-run cells independently — each one below creates and closes its own driver.

## 🎯 Key Takeaways

- Selenium automates a **real browser** — needed for JavaScript-rendered pages.
- `webdriver.Chrome()` + `driver.get(url)` starts automation; `driver.quit()` ends it.
- Find elements with `By.ID / CLASS_NAME / CSS_SELECTOR / XPATH`; `find_elements` returns all matches.
- `click()`, `send_keys()`, and `submit()` cover most interactions.
- `WebDriverWait(...).until(EC....)` waits for dynamic content properly.
- Logins are just fill + click — but credentials must come from the user at runtime, never from code.
- Bound your loops: a `while True` in a notebook can lock the kernel forever.

## 🏋️ Practice Exercises

1. Modify the countries cell to print the first **10** countries in reverse alphabetical order.
2. On the forms page, search for `"New York"` instead of `"Boston"` — how many rows come back?
3. Use `driver.save_screenshot("page.png")` after loading the simple page, then open the image.
4. Extend the Cookie Clicker cell to 500 clicks and print the cookie count from the page (`#cookies` element's text).
5. Navigate to a site of your choice and collect the `href` of every link with `driver.find_elements(By.TAG_NAME, "a")` — print the first 10.

## 🚀 Next Steps

- Combine Selenium with the scraping skills from **`05_Web_Scraping`**: automate a login, then scrape the *rendered* data.
- Analyze your scraped results with **`03_Pandas_Data_Analysis`**.
- Build end-to-end automation: collect data on a schedule and save it with the file tools from **`04_File_Handling_and_IO`**.